# 13 — Prototype: bar-frequency volatility HMM

**Development-only prototype.** This notebook implements the first Module 13 switching test after discussions/010–011.

The main design changes relative to the pre-Module-13 sketch are:

1. **30m/60m overlapping targets are not HMM emissions.** The primary emission (block 6) is the *non-overlapping* 30m realized variance $y_t=\log(\sum_{i<6} r_{t-i}^2+\epsilon)$, with HAR windows ending at $t-6$; each 30-minute phase is its own chain of non-overlapping blocks (phases pooled for estimation, every origin filtered on its own chain). The single-bar emission $\log(r_t^2+\epsilon)$ (block 1) is kept as a diagnostic: its log-$\chi^2_1$ noise is so skewed that, on a synthetic process with **no** regimes, intercept switching split bars into states 4 log-points apart with stay probability 0.84 (discussion 013).
2. **Student-t emissions from the start.** A Gaussian scale-switching HMM can create a fake high-volatility state merely to absorb fat tails.
3. **Four non-switching nulls.** Switching is compared with a static and an expanding (session-by-session) K=1 bar-HAR-t, **and** with Module 12's direct 30m/60m HAR + clock model M1, static and expanding. M1 is the strongest non-switching forecaster we have; an iterated bar model that only beats its own K=1 version has not shown that switching adds forecasting value (discussion 013).
4. **K=2 only.** K-selection and HSMM durations are not tested until K=2 earns OOS value.
5. **Same Module 12 economic targets.** The bar model recursively produces expected 30m/60m integrated variance and is scored with QLIKE on the strict Module 12 index-RV targets.

Models:

- **K1**: all mean coefficients shared; one state.
- **S1**: K=2, state-specific scale only. On a log r² emission this switches the *dispersion/shape* of log r², not the volatility level, and can be driven by flat or stale index bars; it is a diagnostic, not a volatility-regime test (013).
- **S2**: K=2, state-specific intercept + scale.
- **S3**: K=2, state-specific HAR persistence + intercept + scale; clock/weekend shared.

No state is given an economic label in this notebook. Any candidate switching gain must later beat a HAR-GARCH-t no-regime simulation null with at least 200 paths before state interpretation.


In [ ]:
from pathlib import Path
import os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 260)
pd.set_option("display.max_rows", 1000)

load_dotenv()
DATA_ROOT = Path(os.environ["DERIBIT_DATA_ROOT"]).expanduser().resolve()
DB_PATH = DATA_ROOT / "database" / "research.duckdb"
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.features import regime_inputs as ri
from data_engineering.modeling import vol_targets as vt
from data_engineering.modeling import vol_har as vh
from data_engineering.modeling import vol_hmm as vhm
from data_engineering.modeling.hsmm.session_eval import paired_sessions

ASSETS = ["AVAX", "BTC", "ETH", "HYPE", "SOL", "XRP"]
DEV_START, SCREEN_TRAIN_END, DEV_END = "2026-09-22", "2026-09-28", "2026-10-07"
HORIZONS = list(vt.HORIZONS)
# Emission block in bars: 6 = non-overlapping 30m RV (primary, discussion 013); 1 = single-bar log r^2 (diagnostic).
BLOCKS = [int(b) for b in os.getenv("M13_BLOCKS", "6,1").split(",")]
def tag(name, b):
    return f"{name}@b{b}"
M12_BENCHMARKS = ["M12_M1", "M12_M1_ADAPT"]
def benchmarks(b):
    return [tag("STATIC_K1", b), tag("ADAPT_K1", b)] + M12_BENCHMARKS
from types import SimpleNamespace
N_BOOT = int(os.getenv("M13_BOOT", "2000"))
N_INIT = int(os.getenv("M13_N_INIT", "4"))
MAX_ITER = int(os.getenv("M13_MAX_ITER", "150"))
SEED = int(os.getenv("M13_SEED", "1300"))

if vhm.VOL_HMM_VERSION < "0.2.0":
    raise RuntimeError("Old vol_hmm: git pull and restart the kernel.")
print(f"vol_hmm {vhm.VOL_HMM_VERSION} | blocks {BLOCKS} | n_init {N_INIT} | max_iter {MAX_ITER} | boot {N_BOOT}")


## 1. Development data only

Module 13 specification work is restricted to sessions through 2026-10-07. Module 12's frozen confirmation specification is untouched.


In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
cols = set(con.execute("DESCRIBE time_series_feature_state").df()["column_name"])
required = {"state_id", "state_time", "underlying", "return_5m", "futures_index_price", "is_weekend", "in_settlement_window", "option_atm_quote_ok", "option_iv_atm_forward"}
missing = sorted(required - cols)
if missing:
    raise RuntimeError(f"time_series_feature_state lacks {missing}")

DEV_LAST_BAR = pd.Timestamp(f"{DEV_END} 08:00", tz="UTC")
want = sorted(required)
df = con.execute(
    f"SELECT {', '.join(want)} FROM time_series_feature_state "
    f"WHERE state_time <= TIMESTAMPTZ '{DEV_LAST_BAR.isoformat()}' "
    "ORDER BY underlying, state_time"
).df()
con.close()

df["state_time"] = pd.to_datetime(df["state_time"], utc=True)
df = df[df["underlying"].isin(ASSETS)].reset_index(drop=True)
df["session_date"] = ri.session_key(df["state_time"])
keep = ((df["session_date"] >= DEV_START) & (df["session_date"] <= DEV_END)) | (df["state_time"] == DEV_LAST_BAR)
df = df[keep].reset_index(drop=True)

DEV_SESSIONS = sorted(s for s in df["session_date"].unique() if DEV_START <= s <= DEV_END)
SCREEN_TRAIN = {s for s in DEV_SESSIONS if s <= SCREEN_TRAIN_END}
SCREEN_EVAL = {s for s in DEV_SESSIONS if s > SCREEN_TRAIN_END}

train_weekend = df["session_date"].isin(SCREEN_TRAIN) & df["is_weekend"].fillna(False).astype(bool)
if not train_weekend.any():
    raise RuntimeError("Initial Module 13 training set must contain weekend observations.")

print(f"rows {len(df):,} | development sessions {len(DEV_SESSIONS)} | train {len(SCREEN_TRAIN)} | eval {len(SCREEN_EVAL)}")
print("train:", sorted(SCREEN_TRAIN))
print("eval :", sorted(SCREEN_EVAL))


## 2. Fit static K1 and K=2 S1/S2/S3; build expanding K1 benchmark

The K=2 models are deliberately trained only on the initial 2026-09-22 → 09-28 window. The adaptive K1 benchmark is re-estimated before each evaluation session using every completed earlier development session. This makes the switching gate conservative: a static HMM must beat a non-switching model that is allowed to adapt to level drift.

Module 12's direct HAR + clock **M1** (v0.2 backward coverage, target epsilon fixed to the same value as the bar models so QLIKE is on one scale) is added, static and expanding. Its forecast of $RV+\epsilon$ is converted to the same scoring convention.


In [ ]:
def asset_frame(asset):
    return df[df["underlying"] == asset].sort_values("state_time").reset_index(drop=True)


def m12_m1_frame(g, hmm_panel, train_sessions, eval_sessions, name, score_eps):
    """Module 12 v0.2 direct M1 (HAR + clock) scored on the bar-model convention."""
    P = vt.build_panel(g)
    P.back_rv = {w: vhm._backward_sum_coverage(P.r ** 2, P.times, k) for w, k in vt.BACK_WINDOWS.items()}
    nuis = vh.fit_nuisance(g, P, train_sessions)
    nuis.eps_target = {h: float(score_eps[h]) for h in vt.HORIZONS}      # one QLIKE scale for every model
    mat = vh.materialize(P, nuis, pd.DataFrame(index=np.arange(len(g))))
    tr = np.isin(mat.sessions, sorted(train_sessions))
    raw = np.flatnonzero(np.isin(mat.sessions, sorted(eval_sessions)))
    out = []
    for h in vt.HORIZONS:
        fv = vh.fit_vol(mat, h, "M1", [], tr)
        level = np.exp(vh.predict_log(fv, mat, h)) * fv.smear           # forecast of RV + eps
        out.append(vhm.score_forecasts(hmm_panel, SimpleNamespace(raw_index=raw), level[raw] - score_eps[h], h, name))
    return out


STATIC_FITS, TRAIN_DATA = {}, {}
SCORE_FRAMES, STATE_ROWS, EPISODE_ROWS, DIAG_ROWS = [], [], [], []

t_all = time.perf_counter()
for ai, asset in enumerate(ASSETS):
    g = asset_frame(asset)
    base_panel = vhm.build_panel(g, SCREEN_TRAIN, block=1)
    score_eps = dict(base_panel.score_eps)

    # Module 12 direct M1: static (initial window) and expanding (before each session).
    for sc in m12_m1_frame(g, base_panel, SCREEN_TRAIN, SCREEN_EVAL, "M12_M1", score_eps):
        sc.insert(0, "underlying", asset)
        SCORE_FRAMES.append(sc)
    for session in sorted(SCREEN_EVAL):
        prior = {s for s in DEV_SESSIONS if s < session}
        for sc in m12_m1_frame(g, base_panel, prior, {session}, "M12_M1_ADAPT", score_eps):
            sc.insert(0, "underlying", asset)
            SCORE_FRAMES.append(sc)

    for b in BLOCKS:
        panel = vhm.build_panel(g, SCREEN_TRAIN, score_eps=score_eps, block=b)

        # Static one-state benchmark and static K=2 switching prototypes.
        for mi, structure in enumerate(["K1", "S1", "S2", "S3"]):
            fit, tr_data = vhm.fit_model(
                panel, SCREEN_TRAIN, structure,
                K=1 if structure == "K1" else 2,
                seed=SEED + 100 * ai + 10 * b + mi,
                n_init=N_INIT,
                max_iter=MAX_ITER,
            )
            name = tag("STATIC_K1" if structure == "K1" else structure, b)
            STATIC_FITS[(asset, name)] = fit
            TRAIN_DATA[(asset, name)] = tr_data

            pt = vhm.parameter_table(fit)
            pt.insert(0, "underlying", asset)
            pt.insert(1, "model", name)
            STATE_ROWS.append(pt)

            dg = vhm.state_emission_diagnostics(fit, tr_data, panel)
            dg.insert(0, "underlying", asset)
            dg.insert(1, "model", name)
            DIAG_ROWS.append(dg)

            ep = vhm.episode_table(fit, tr_data)
            if len(ep):
                es = ep.groupby("state").agg(
                    episodes=("length", "size"),
                    median_length=("length", "median"),
                    mean_length=("length", "mean"),
                ).reset_index()
                es.insert(0, "underlying", asset)
                es.insert(1, "model", name)
                EPISODE_ROWS.append(es)

            ev_data, filt = vhm.filter_model(fit, panel, SCREEN_EVAL)
            for h, m in vt.HORIZONS.items():
                f = vhm.forecast_integrated_variance(fit, panel, ev_data, filt, m)
                sc = vhm.score_forecasts(panel, ev_data, f, h, name)
                sc.insert(0, "underlying", asset)
                SCORE_FRAMES.append(sc)

        # Expanding one-state benchmark for this emission. Refit only before each new session.
        for si, session in enumerate(sorted(SCREEN_EVAL)):
            prior = {s for s in DEV_SESSIONS if s < session}
            p_ad = vhm.build_panel(g, prior, score_eps=score_eps, block=b)
            fit_ad, _ = vhm.fit_model(
                p_ad, prior, "K1", K=1,
                seed=SEED + 10000 + 100 * ai + 10 * b + si,
                n_init=1,
                max_iter=MAX_ITER,
            )
            ev_ad, filt_ad = vhm.filter_model(fit_ad, p_ad, {session})
            for h, m in vt.HORIZONS.items():
                f = vhm.forecast_integrated_variance(fit_ad, p_ad, ev_ad, filt_ad, m)
                sc = vhm.score_forecasts(p_ad, ev_ad, f, h, tag("ADAPT_K1", b))
                sc.insert(0, "underlying", asset)
                SCORE_FRAMES.append(sc)

    print(f"{asset}: complete in {time.perf_counter() - t_all:.0f}s cumulative")

scores = pd.concat(SCORE_FRAMES, ignore_index=True)
state_params = pd.concat(STATE_ROWS, ignore_index=True)
episodes = pd.concat(EPISODE_ROWS, ignore_index=True) if EPISODE_ROWS else pd.DataFrame()
emission_diag = pd.concat(DIAG_ROWS, ignore_index=True)
print(f"score rows: {len(scores):,}")


## 3. Descriptive forecast metrics

These are development-screening metrics. The switching gate is the paired comparison in the next section, not the standalone R²/QLIKE table.


In [ ]:
metrics = (
    scores.dropna(subset=["qlike"])
    .groupby(["underlying", "horizon", "model"])
    .agg(
        n=("qlike", "size"),
        qlike=("qlike", "mean"),
        log_rmse=("sqerr", lambda x: float(np.sqrt(np.mean(x)))),
    )
    .reset_index()
)
display(metrics.round(6))

## 4. Session-paired switching gate

Positive QLIKE difference means the first model is better. A candidate switching result must beat **all four** non-switching benchmarks: STATIC_K1, ADAPT_K1, M12_M1 and M12_M1_ADAPT. The non-overlapping score view must agree in sign; a bootstrap interval above zero is reported but remains development evidence only.


In [ ]:
def compare_pair(asset, horizon, a_model, b_model, view):
    A = scores[(scores.underlying == asset) & (scores.horizon == horizon) & (scores.model == a_model)].copy()
    B = scores[(scores.underlying == asset) & (scores.horizon == horizon) & (scores.model == b_model)].copy()
    d = A.merge(B, on=["underlying", "raw_index", "session", "horizon"], suffixes=("_a", "_b"))
    ok = np.isfinite(d["qlike_a"]) & np.isfinite(d["qlike_b"])
    if view == "non_overlapping":
        step = vt.HORIZONS[horizon] * vt.BAR_MIN
        ok &= np.isclose(np.mod(np.round(d["tte_min_a"].to_numpy()), step), 0.0)
    d = d.loc[ok].copy()
    if d.empty:
        return {"n_rows": 0, "n_sessions": 0, "mean_session_diff": np.nan, "boot_lo": np.nan, "boot_hi": np.nan, "sessions_won": np.nan}
    diff = d["qlike_b"].to_numpy() - d["qlike_a"].to_numpy()
    p = paired_sessions(diff, np.zeros_like(diff), d["session"].to_numpy(), n_boot=N_BOOT)
    return {
        "n_rows": len(d),
        "n_sessions": p["n_sessions"],
        "mean_session_diff": p["mean_session_diff"],
        "boot_lo": p["boot_lo"],
        "boot_hi": p["boot_hi"],
        "sessions_won": p["sessions_won"],
    }

pairs = []
for asset in ASSETS:
    for h in HORIZONS:
        for b in BLOCKS:
            for model in [tag(m, b) for m in ("S1", "S2", "S3")]:
                for benchmark in benchmarks(b):
                    for view in ["all", "non_overlapping"]:
                        pairs.append({"underlying": asset, "horizon": h, "block": b, "model": model, "vs": benchmark, "view": view,
                                      **compare_pair(asset, h, model, benchmark, view)})
            for model, benchmark in [(tag("S2", b), tag("S1", b)), (tag("S3", b), tag("S2", b)),
                                     (tag("STATIC_K1", b), "M12_M1"), (tag("ADAPT_K1", b), "M12_M1_ADAPT")]:
                for view in ["all", "non_overlapping"]:
                    pairs.append({"underlying": asset, "horizon": h, "block": b, "model": model, "vs": benchmark, "view": view,
                                  **compare_pair(asset, h, model, benchmark, view)})

comparisons = pd.DataFrame(pairs)
display(comparisons.round(6))

OUT = PROJECT_ROOT / "notebooks"
metrics.to_csv(OUT / "13_screening_metrics.csv", index=False)
comparisons.to_csv(OUT / "13_screening_comparisons.csv", index=False)
state_params.to_csv(OUT / "13_state_parameters.csv", index=False)
episodes.to_csv(OUT / "13_state_episodes.csv", index=False)
emission_diag.to_csv(OUT / "13_state_emission_diagnostics.csv", index=False)
print("wrote Module 13 prototype CSV outputs")


## 5. Anonymous state diagnostics

These diagnostics describe the fitted training states only. `floor_share` is the share of a state's bars at the epsilon floor ($r^2\le\epsilon$): a state dominated by floor bars describes flat or stale index prints (a Q-like observation effect), not a volatility regime. Do not call them “low vol”, “shock”, MR, MOM, or Bridge yet. The first legitimate labels come only after the switching forecast gate and the no-regime null pass.


In [ ]:
display(state_params.round(5))
display(emission_diag.round(4))
if len(episodes):
    display(episodes.round(3))


## 6. Prototype gate and next required test

A model/horizon is only a **candidate for the no-regime null** if its mean session QLIKE improvement is positive against all four non-switching benchmarks in both score views. Fits with a state below 2% occupancy or at the sigma floor are flagged `degenerate_fit` and cannot be candidates. Only the primary block-6 emission counts; block-1 passes are diagnostics. Only S2/S3 (intercept switching) can be candidates for a *volatility-level* regime; an S1-only pass is reported as a dispersion/observation effect. This deliberately weak development gate does not declare a regime.

If any candidate appears, the next implementation step is the mandatory HAR-GARCH-t simulation null from discussion/011 §5.8, with at least 200 paths and the real missingness/session/clock pattern. If no candidate appears, the switching branch stops before HSMM.


In [ ]:
gate_rows = []
for asset in ASSETS:
    for h in HORIZONS:
        for b in BLOCKS:
            for base in ["S1", "S2", "S3"]:
                model = tag(base, b)
                d = comparisons[(comparisons.underlying == asset) & (comparisons.horizon == h) & (comparisons.model == model)
                                & (comparisons["vs"].isin(benchmarks(b)))]
                tests = {(r["vs"], r["view"]): r for _, r in d.iterrows()}
                required = [(bm, v) for bm in benchmarks(b) for v in ["all", "non_overlapping"]]
                pass_sign = all(k in tests and np.isfinite(tests[k]["mean_session_diff"]) and tests[k]["mean_session_diff"] > 0 for k in required)
                pass_ci = all(k in tests and np.isfinite(tests[k]["boot_lo"]) and tests[k]["boot_lo"] > 0 for k in required)
                degen = bool(state_params[(state_params.underlying == asset) & (state_params.model == model)]["degenerate_fit"].any())
                gate_rows.append({"underlying": asset, "horizon": h, "block": b, "model": model,
                                  "primary_emission": b == 6, "level_regime_test": base != "S1", "degenerate_fit": degen,
                                  "positive_vs_all_benchmarks": pass_sign, "ci_above_zero_vs_all": pass_ci})

gates = pd.DataFrame(gate_rows)
display(gates)
cand = gates[gates["positive_vs_all_benchmarks"] & gates["level_regime_test"] & ~gates["degenerate_fit"] & gates["primary_emission"]]
diag_pass = gates[gates["positive_vs_all_benchmarks"] & ~(gates["level_regime_test"] & ~gates["degenerate_fit"] & gates["primary_emission"])]
if len(diag_pass):
    print("Passes that do NOT count (S1 dispersion-only, degenerate fit, or single-bar diagnostic emission):")
    display(diag_pass)
if len(cand):
    print("Candidate switching cases (block-6 emission, S2/S3) require the HAR-GARCH-t null >= 200 paths before interpretation:")
    display(cand)
else:
    print("No block-6 S2/S3 switching structure beats all four non-switching benchmarks in both score views. Do not proceed to HSMM.")


## 7. Prototype limitations

This first implementation intentionally stops before three expensive extensions:

1. **HAR-GARCH-t no-regime null** — mandatory only for candidates that survive §6. It must simulate **returns** $r_t=\sigma_t z_t$ (log-variance HAR + clock, GARCH on the log-variance innovation, $t$-distributed $z_t$) and rebuild $\log(r_t^2+\epsilon)$, so the null carries the same log-$\chi^2$-type emission noise as the data; simulating Student-t residuals directly in log space would not (013).
2. **Adaptive refitting of K=2 HMMs** — the prototype gives switching a harder test by keeping K=2 static while allowing K1 to adapt. If a static K=2 candidate survives, a like-for-like expanding HMM can be added before the null.
3. **HSMM duration** — not tested until a K=2 HMM survives the two non-switching benchmarks and the no-regime null.

Module 12's `12_frozen_spec_v02.json` remains untouched throughout.
